# Entropy-Adaptive Background Job Scheduler

**Research Question:** Can we use Shannon entropy of CPU load to better predict and protect against scheduling conflicts?

This notebook implements a comparative experiment evaluating 4 scheduler variants:
1. **Baseline**: Fixed 55% CPU threshold
2. **Entropy-Only**: Adaptive threshold based on load entropy
3. **Momentum-Only**: Threshold with momentum-aware dispatch
4. **Entropy-Adaptive**: Combined entropy + momentum (proposed)

Each scheduler is tested on synthetic bursty workloads and evaluated on:
- Job completion variance (coefficient of variation)
- Foreground task p95 latency protection
- False positive rate (scheduling when load spikes imminent)
- Entropy-outcome correlation

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# psutil, loguru — NOT pre-installed on Colab, always install
_pip('psutil==6.1.0')
_pip('loguru==0.7.2')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import json
import sys
import os
import math
import time
import random
import gc
import threading
import queue
import psutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import List, Dict, Optional, Tuple
from loguru import logger
from scipy import stats
from collections import deque

# Suppress verbose logging
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
# Data loading helper with GitHub URL fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-b6315a-entropy-adaptive-background-job/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_demo_data():
    """Load demo data from GitHub URL or local file."""
    # Try GitHub first
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=5) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    
    # Fallback to local file
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    # If neither works, return empty structure
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local file")

In [ ]:
# Load the demo data
demo_data = load_demo_data()
print(f"Loaded demo data: {demo_data['metadata']}")
print(f"Number of examples: {len(demo_data['datasets'][0]['examples'])}")

## Configuration

All tunable parameters are defined here. For the demo, we use **MINIMUM VALUES** that complete quickly while showing the approach. To run the full experiment, gradually increase these values.

In [ ]:
# ========== TUNABLE PARAMETERS (set to minimum for demo) ==========

# Trial and timing
N_TRIALS = 2                    # Minimum: 2 (scale up to 50 for full experiment)
TRIAL_DURATION_SEC = 5          # Minimum: 5s per trial (original: 60s)

# Entropy monitor
ENTROPY_WINDOW_SIZE_SEC = 10    # Minimum: 10s window
ENTROPY_SAMPLE_INTERVAL_SEC = 1.0  # Minimum: 1s sample interval
ENTROPY_NUM_BINS = 4            # Minimum: 4 bins (original: 8)

# Momentum tracker
MOMENTUM_ALPHA = 0.2
MOMENTUM_UPDATE_INTERVAL_SEC = 1.0

# Workload configs
WORKLOAD_CONFIGS = [
    {'arrival_rate': 5, 'has_spikes': True, 'spike_freq_sec': 10, 'spike_duration_sec': 2},   # Bursty
    {'arrival_rate': 8, 'has_spikes': False},  # Steady
]

# Background job executor
JOB_SPAWN_INTERVAL = 5.0        # Spawn one job every ~5 seconds
JOB_DURATION_RANGE = (2, 5)     # Minimum: 2-5 sec job duration (original: 10-100s)
METRIC_COLLECTION_INTERVAL = 2.0  # Collect metrics every 2s

print("✓ Configuration loaded:")
print(f"  N_TRIALS = {N_TRIALS}")
print(f"  TRIAL_DURATION_SEC = {TRIAL_DURATION_SEC}s")
print(f"  Workloads: {len(WORKLOAD_CONFIGS)} variants")

## Entropy Monitor

Monitors CPU load over a sliding window and computes normalized Shannon entropy.
High entropy = unpredictable load = higher threshold needed.
Low entropy = predictable load = can schedule more aggressively.

In [ ]:
class EntropyMonitor:
    """Shannon entropy estimator of CPU load unpredictability."""

    def __init__(self, window_size_sec: int = 60, sample_interval_sec: float = 0.5,
                 num_bins: int = 8):
        self.window_size_sec = window_size_sec
        self.sample_interval_sec = sample_interval_sec
        self.num_bins = num_bins
        self.max_samples = int(window_size_sec / sample_interval_sec)
        self.load_samples = deque(maxlen=self.max_samples)
        self.last_sample_time = None

    def sample_cpu_load(self) -> float:
        """Get current CPU utilization percent (0-100)."""
        return psutil.cpu_percent(interval=0.05, percpu=False)

    def update(self) -> None:
        """Add new CPU sample if enough time has passed."""
        now = time.time()
        if self.last_sample_time is None or (now - self.last_sample_time) >= self.sample_interval_sec:
            load = self.sample_cpu_load()
            self.load_samples.append(load)
            self.last_sample_time = now

    def compute_entropy(self) -> float:
        """Compute normalized Shannon entropy with Miller-Madow bias correction."""
        if len(self.load_samples) < 2:
            return 0.0

        samples = list(self.load_samples)
        min_load, max_load = min(samples), max(samples)

        if min_load == max_load:
            return 0.0

        bin_edges = np.percentile(samples, np.linspace(0, 100, self.num_bins + 1))
        bin_edges[0] = min_load - 0.001
        bin_edges[-1] = max_load + 0.001

        counts, _ = np.histogram(samples, bins=bin_edges)
        counts = counts[counts > 0]

        if len(counts) == 0:
            return 0.0

        probs = counts / len(samples)
        h_naive = -np.sum(probs * np.log2(np.maximum(probs, 1e-10)))

        k = len(counts)
        n = len(samples)
        h_corrected = h_naive + (k - 1) / (2 * n)

        h_norm = h_corrected / np.log2(self.num_bins)
        return min(1.0, max(0.0, h_norm))

print("✓ EntropyMonitor class defined")

## Momentum Tracker

Tracks CPU load trend using exponential weighted moving average.
Helps detect if load is rising (risky to schedule) or falling (safe to schedule).

In [ ]:
class MomentumTracker:
    """Exponential weighted moving average of CPU load trend."""

    def __init__(self, alpha: float = 0.2, update_interval_sec: float = 1.0):
        self.alpha = alpha
        self.update_interval_sec = update_interval_sec
        self.momentum = None
        self.last_load = None
        self.last_update_time = None

    def update(self, current_load: float) -> None:
        """Update EWMA momentum."""
        now = time.time()
        if self.last_update_time is None or (now - self.last_update_time) >= self.update_interval_sec:
            if self.momentum is None:
                self.momentum = current_load
            else:
                self.momentum = self.alpha * current_load + (1 - self.alpha) * self.momentum
            self.last_load = current_load
            self.last_update_time = now

    def get_gradient(self) -> float:
        """Compute load gradient: (load_t - momentum_t) / momentum_t."""
        if self.momentum is None or self.momentum < 1e-6:
            return 0.0
        return (self.last_load - self.momentum) / self.momentum

    def is_negative(self) -> bool:
        """Load declining (< -5%)."""
        return self.get_gradient() < -0.05

    def is_positive(self) -> bool:
        """Load rising (> +10%)."""
        return self.get_gradient() > 0.10

print("✓ MomentumTracker class defined")

## Scheduler Variants

Four scheduling strategies:
1. **Baseline**: Dumb fixed threshold (55%)
2. **EntropyOnly**: Adapt threshold based on entropy
3. **MomentumOnly**: Use momentum cues within fixed threshold
4. **EntropyAdaptive**: Combined (the proposal)

In [ ]:
@dataclass
class SchedulingDecision:
    schedule: bool
    priority: str  # 'STANDARD' or 'PRIORITY'
    threshold_pct: float
    reason: str

class BaseScheduler:
    """Base class for all scheduler variants."""
    def __init__(self, name: str, entropy_monitor: 'EntropyMonitor',
                 momentum_tracker: 'MomentumTracker'):
        self.name = name
        self.entropy_monitor = entropy_monitor
        self.momentum_tracker = momentum_tracker
    def decide(self, current_cpu_pct: float) -> SchedulingDecision:
        raise NotImplementedError

class BaselineScheduler(BaseScheduler):
    """Fixed 55% CPU threshold."""
    def decide(self, current_cpu_pct: float) -> SchedulingDecision:
        threshold = 55.0
        if current_cpu_pct < threshold:
            return SchedulingDecision(True, "STANDARD", threshold, "CPU below fixed 55%")
        else:
            return SchedulingDecision(False, "NONE", threshold, "CPU above fixed 55%")

class EntropyOnlyScheduler(BaseScheduler):
    """Adaptive threshold based on entropy only."""
    def decide(self, current_cpu_pct: float) -> SchedulingDecision:
        h_norm = self.entropy_monitor.compute_entropy()
        threshold = 70.0 - (70.0 - 25.0) * h_norm
        if current_cpu_pct < threshold:
            return SchedulingDecision(True, "STANDARD", threshold, f"Entropy-adaptive: H={h_norm:.2f}")
        else:
            return SchedulingDecision(False, "NONE", threshold, f"Entropy high: H={h_norm:.2f}")

class MomentumOnlyScheduler(BaseScheduler):
    """Momentum-aware dispatch with fixed 55% threshold."""
    def decide(self, current_cpu_pct: float) -> SchedulingDecision:
        threshold = 55.0
        if current_cpu_pct >= threshold:
            return SchedulingDecision(False, "NONE", threshold, "CPU above 55%")
        if self.momentum_tracker.is_negative():
            return SchedulingDecision(True, "PRIORITY", threshold, "Negative gradient detected")
        else:
            return SchedulingDecision(True, "STANDARD", threshold, "CPU below 55%")

class EntropyAdaptiveScheduler(BaseScheduler):
    """Combined entropy + momentum scheduling (proposed)."""
    def decide(self, current_cpu_pct: float) -> SchedulingDecision:
        h_norm = self.entropy_monitor.compute_entropy()
        threshold = 70.0 - (70.0 - 25.0) * h_norm
        if current_cpu_pct >= threshold:
            return SchedulingDecision(False, "NONE", threshold, "CPU above threshold")
        if h_norm < 0.6 and self.momentum_tracker.is_negative():
            return SchedulingDecision(True, "PRIORITY", threshold, f"Entropy low + negative gradient")
        else:
            return SchedulingDecision(True, "STANDARD", threshold, f"Standard dispatch (H={h_norm:.2f})")

print("✓ All 4 scheduler variants defined")

## Support Classes

ForegroundEmulator (simulates user-facing requests) and BackgroundJobExecutor (manages job queue).

In [ ]:
@dataclass
class ForegroundRequest:
    arrival_time: float
    service_time_sec: float
    completion_time: Optional[float] = None

class ForegroundEmulator:
    """Poisson arrivals with lognormal service times."""
    def __init__(self, arrival_rate_per_sec: float = 15.0):
        self.arrival_rate = arrival_rate_per_sec
        self.requests = []
        self.completed_requests = []
        self.start_time = time.time()

    def generate_arrivals(self, current_time: float, duration_sec: float) -> List[ForegroundRequest]:
        arrivals = []
        lambda_param = self.arrival_rate * duration_sec
        num_arrivals = np.random.poisson(lambda_param)
        for _ in range(num_arrivals):
            arrival_time = self.start_time + current_time + np.random.uniform(0, duration_sec)
            service_time = np.random.lognormal(mean=np.log(0.01), sigma=0.5)
            arrivals.append(ForegroundRequest(arrival_time, service_time))
        self.requests.extend(arrivals)
        return arrivals

    def process_requests(self, current_time: float) -> None:
        for req in self.requests:
            if req.completion_time is None and req.arrival_time + req.service_time_sec <= self.start_time + current_time:
                req.completion_time = req.arrival_time + req.service_time_sec
                self.completed_requests.append(req)

    def get_latencies_ms(self) -> List[float]:
        return [(r.completion_time - r.arrival_time) * 1000
                for r in self.completed_requests if r.completion_time is not None]

    def get_percentile_latency(self, percentile: float) -> Optional[float]:
        latencies = self.get_latencies_ms()
        if len(latencies) == 0:
            return None
        return np.percentile(latencies, percentile)

@dataclass
class BackgroundJob:
    job_id: int
    queued_at: float
    duration_sec: float
    started_at: Optional[float] = None
    completed_at: Optional[float] = None

    @property
    def completion_time_sec(self) -> Optional[float]:
        if self.completed_at is not None:
            return self.completed_at - self.queued_at
        return None

class BackgroundJobExecutor:
    """Manages background job queue and execution."""
    def __init__(self, job_duration_range: Tuple[float, float] = (10, 100)):
        self.job_queue = queue.Queue()
        self.job_id_counter = 0
        self.completed_jobs = []
        self.job_duration_range = job_duration_range
        self.current_job = None
        self.current_job_start = None

    def enqueue_job(self, current_time: float) -> BackgroundJob:
        duration = np.random.uniform(*self.job_duration_range)
        job = BackgroundJob(self.job_id_counter, current_time, duration)
        self.job_id_counter += 1
        self.job_queue.put(job)
        return job

    def try_schedule_job(self, current_time: float, scheduler_decision: SchedulingDecision) -> bool:
        if not scheduler_decision.schedule or self.job_queue.empty():
            return False
        job = self.job_queue.get()
        job.started_at = current_time
        self.current_job = job
        self.current_job_start = current_time
        return True

    def try_complete_job(self, current_time: float) -> bool:
        if self.current_job is None:
            return False
        if current_time - self.current_job_start >= self.current_job.duration_sec:
            self.current_job.completed_at = current_time
            self.completed_jobs.append(self.current_job)
            self.current_job = None
            self.current_job_start = None
            return True
        return False

    def get_completion_times(self) -> List[float]:
        return [j.completion_time_sec for j in self.completed_jobs if j.completion_time_sec is not None]

print("✓ Support classes defined")

## Workload Generator

Simulates bursty traffic with optional load spikes.

In [ ]:
class WorkloadConfig:
    """Configuration for a trial's workload."""
    def __init__(self, arrival_rate: int = 15, has_spikes: bool = False,
                 spike_freq_sec: int = 60, spike_duration_sec: int = 10):
        self.arrival_rate = arrival_rate
        self.has_spikes = has_spikes
        self.spike_freq_sec = spike_freq_sec
        self.spike_duration_sec = spike_duration_sec
        self.current_spike_intensity = 1.0

    def update(self, current_time: float) -> None:
        """Update workload intensity based on spike schedule."""
        if not self.has_spikes:
            self.current_spike_intensity = 1.0
            return
        cycle_pos = current_time % self.spike_freq_sec
        if cycle_pos < self.spike_duration_sec:
            self.current_spike_intensity = 2.0
        else:
            self.current_spike_intensity = 1.0

    def get_current_rate(self) -> float:
        return self.arrival_rate * self.current_spike_intensity

print("✓ WorkloadConfig defined")

## Metrics and Trial Execution

TrialMetrics aggregates results per trial, and Trial runs one full experiment.

In [ ]:
@dataclass
class TrialMetrics:
    trial_id: int
    scheduler_name: str
    completion_time_mean: float
    completion_time_std: float
    completion_time_cv: float
    completion_count: int
    foreground_p50_ms: Optional[float]
    foreground_p95_ms: Optional[float]
    foreground_p99_ms: Optional[float]
    false_positive_rate: float
    entropy_samples: List[float] = field(default_factory=list)
    momentum_samples: List[float] = field(default_factory=list)

class Trial:
    """Single trial of the experiment."""
    def __init__(self, trial_id: int, scheduler: BaseScheduler, workload_config: WorkloadConfig,
                 trial_duration_sec: int = 600):
        self.trial_id = trial_id
        self.scheduler = scheduler
        self.workload_config = workload_config
        self.trial_duration_sec = trial_duration_sec
        self.start_time = None
        self.entropy_monitor = scheduler.entropy_monitor
        self.momentum_tracker = scheduler.momentum_tracker
        self.foreground = ForegroundEmulator(workload_config.arrival_rate)
        self.bg_executor = BackgroundJobExecutor(JOB_DURATION_RANGE)
        self.metric_log = []
        self.entropy_samples = []
        self.momentum_samples = []
        self.false_positive_count = 0
        self.false_positive_total = 0

    def run(self) -> TrialMetrics:
        """Execute trial."""
        self.start_time = time.time()
        last_job_spawn = 0
        last_metric_collection = 0
        
        logger.info(f"[Trial {self.trial_id}] Starting {self.scheduler.name} for {self.trial_duration_sec}s")

        while True:
            elapsed = time.time() - self.start_time
            if elapsed >= self.trial_duration_sec:
                break

            # Update workload and sample CPU
            self.workload_config.update(elapsed)
            current_cpu = psutil.cpu_percent(interval=0.01, percpu=False)
            self.entropy_monitor.update()
            self.momentum_tracker.update(current_cpu)

            # Generate foreground arrivals
            if elapsed - last_metric_collection >= 0.5:
                self.foreground.generate_arrivals(elapsed, 0.5)
                self.foreground.process_requests(elapsed)
                last_metric_collection = elapsed

            # Spawn background jobs
            if elapsed - last_job_spawn >= JOB_SPAWN_INTERVAL:
                self.bg_executor.enqueue_job(self.start_time + elapsed)
                last_job_spawn = elapsed

            # Scheduler decision
            decision = self.scheduler.decide(current_cpu)

            # Track false positives
            if decision.schedule:
                self.false_positive_total += 1
                if current_cpu < 50 and self.workload_config.current_spike_intensity > 1.5:
                    self.false_positive_count += 1

            # Schedule and complete jobs
            self.bg_executor.try_schedule_job(self.start_time + elapsed, decision)
            self.bg_executor.try_complete_job(self.start_time + elapsed)

            # Collect metrics
            if elapsed - last_metric_collection >= METRIC_COLLECTION_INTERVAL:
                h = self.entropy_monitor.compute_entropy()
                grad = self.momentum_tracker.get_gradient()
                self.entropy_samples.append(h)
                self.momentum_samples.append(grad)

            time.sleep(0.01)

        # Aggregate metrics
        completion_times = self.bg_executor.get_completion_times()
        if len(completion_times) > 0:
            completion_mean = np.mean(completion_times)
            completion_std = np.std(completion_times)
            completion_cv = completion_std / completion_mean if completion_mean > 0 else 0
        else:
            completion_mean = completion_std = completion_cv = 0

        fpr = self.false_positive_count / max(1, self.false_positive_total)

        metrics = TrialMetrics(
            trial_id=self.trial_id,
            scheduler_name=self.scheduler.name,
            completion_time_mean=completion_mean,
            completion_time_std=completion_std,
            completion_time_cv=completion_cv,
            completion_count=len(completion_times),
            foreground_p50_ms=self.foreground.get_percentile_latency(50),
            foreground_p95_ms=self.foreground.get_percentile_latency(95),
            foreground_p99_ms=self.foreground.get_percentile_latency(99),
            false_positive_rate=fpr,
            entropy_samples=self.entropy_samples,
            momentum_samples=self.momentum_samples,
        )

        logger.info(f"[Trial {self.trial_id}] {self.scheduler.name}: CV={completion_cv:.3f}, P95={metrics.foreground_p95_ms:.1f}ms")
        return metrics

print("✓ Trial and TrialMetrics classes defined")

## Statistical Analysis

Aggregate results across trials and compute effect sizes, confidence intervals, and significance tests.

In [ ]:
def analyze_results(all_metrics: List[TrialMetrics]) -> Dict:
    """Compute aggregate statistics and comparisons."""

    # Group by scheduler
    by_scheduler = {}
    for metric in all_metrics:
        if metric.scheduler_name not in by_scheduler:
            by_scheduler[metric.scheduler_name] = []
        by_scheduler[metric.scheduler_name].append(metric)

    # Per-scheduler stats
    scheduler_stats = {}
    for sched_name, metrics_list in by_scheduler.items():
        cvs = [m.completion_time_cv for m in metrics_list]
        p95s = [m.foreground_p95_ms for m in metrics_list if m.foreground_p95_ms is not None]
        fprs = [m.false_positive_rate for m in metrics_list]

        scheduler_stats[sched_name] = {
            'n_trials': len(metrics_list),
            'mean_cv': float(np.mean(cvs)) if cvs else 0,
            'std_cv': float(np.std(cvs)) if cvs else 0,
            'mean_p95_ms': float(np.mean(p95s)) if p95s else 0,
            'std_p95_ms': float(np.std(p95s)) if p95s else 0,
            'mean_fpr': float(np.mean(fprs)) if fprs else 0,
            'std_fpr': float(np.std(fprs)) if fprs else 0,
        }

    return {
        'schedulers': scheduler_stats,
        'total_trials': len(all_metrics) // 4 if len(all_metrics) > 0 else 0,
    }

print("✓ analyze_results function defined")

## Run Experiment

Execute N_TRIALS with all 4 scheduler variants.

In [ ]:
# Run experiment
logger.info("="*80)
logger.info("ENTROPY-ADAPTIVE SCHEDULER EXPERIMENT (DEMO)")
logger.info("="*80)

all_metrics = []

# Use mini workload config
workload_configs = [
    WorkloadConfig(
        arrival_rate=config['arrival_rate'],
        has_spikes=config['has_spikes'],
        spike_freq_sec=config.get('spike_freq_sec', 60),
        spike_duration_sec=config.get('spike_duration_sec', 10)
    )
    for config in WORKLOAD_CONFIGS
]

for trial_id in range(N_TRIALS):
    workload = workload_configs[trial_id % len(workload_configs)]
    
    # Shared monitors
    entropy_monitor = EntropyMonitor(
        window_size_sec=ENTROPY_WINDOW_SIZE_SEC,
        sample_interval_sec=ENTROPY_SAMPLE_INTERVAL_SEC,
        num_bins=ENTROPY_NUM_BINS
    )
    momentum_tracker = MomentumTracker(
        alpha=MOMENTUM_ALPHA,
        update_interval_sec=MOMENTUM_UPDATE_INTERVAL_SEC
    )
    
    # Schedulers
    schedulers = [
        BaselineScheduler("Baseline", entropy_monitor, momentum_tracker),
        EntropyOnlyScheduler("Entropy-Only", entropy_monitor, momentum_tracker),
        MomentumOnlyScheduler("Momentum-Only", entropy_monitor, momentum_tracker),
        EntropyAdaptiveScheduler("Entropy-Adaptive", entropy_monitor, momentum_tracker),
    ]
    
    # Run trial for each scheduler
    for scheduler in schedulers:
        trial = Trial(trial_id, scheduler, workload, TRIAL_DURATION_SEC)
        metrics = trial.run()
        all_metrics.append(metrics)
        gc.collect()

logger.info(f"\n✓ Completed {N_TRIALS} trials with 4 schedulers each")

## Results Summary

Aggregate metrics and comparison of scheduler variants.

In [ ]:
# Analyze results
results = analyze_results(all_metrics)

print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)
print(f"Total trials: {results['total_trials']}")
print(f"Total scheduler runs: {len(all_metrics)}")
print()

# Print per-scheduler stats
for sched, stats in results['schedulers'].items():
    print(f"\n{sched}:")
    print(f"  Trials: {stats['n_trials']}")
    print(f"  Completion CV: {stats['mean_cv']:.3f} ± {stats['std_cv']:.3f}")
    print(f"  Foreground P95: {stats['mean_p95_ms']:.1f} ± {stats['std_p95_ms']:.1f} ms")
    print(f"  False Positive Rate: {stats['mean_fpr']:.3f} ± {stats['std_fpr']:.3f}")

print("\n" + "="*80)

## Visualization

Compare scheduler performance across key metrics.

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Metric 1: Completion CV
schedulers_list = list(results['schedulers'].keys())
cvs = [results['schedulers'][s]['mean_cv'] for s in schedulers_list]
cv_errs = [results['schedulers'][s]['std_cv'] for s in schedulers_list]

axes[0].bar(range(len(schedulers_list)), cvs, yerr=cv_errs, capsize=5, alpha=0.7, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[0].set_xticks(range(len(schedulers_list)))
axes[0].set_xticklabels(schedulers_list, rotation=45, ha='right')
axes[0].set_ylabel('Completion Time CV')
axes[0].set_title('Job Completion Variance\n(Lower is Better)')
axes[0].grid(axis='y', alpha=0.3)

# Metric 2: P95 Latency
p95s = [results['schedulers'][s]['mean_p95_ms'] for s in schedulers_list]
p95_errs = [results['schedulers'][s]['std_p95_ms'] for s in schedulers_list]

axes[1].bar(range(len(schedulers_list)), p95s, yerr=p95_errs, capsize=5, alpha=0.7, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[1].set_xticks(range(len(schedulers_list)))
axes[1].set_xticklabels(schedulers_list, rotation=45, ha='right')
axes[1].set_ylabel('P95 Latency (ms)')
axes[1].set_title('Foreground Task Protection\n(Lower is Better)')
axes[1].grid(axis='y', alpha=0.3)

# Metric 3: False Positive Rate
fprs = [results['schedulers'][s]['mean_fpr'] for s in schedulers_list]
fpr_errs = [results['schedulers'][s]['std_fpr'] for s in schedulers_list]

axes[2].bar(range(len(schedulers_list)), fprs, yerr=fpr_errs, capsize=5, alpha=0.7, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[2].set_xticks(range(len(schedulers_list)))
axes[2].set_xticklabels(schedulers_list, rotation=45, ha='right')
axes[2].set_ylabel('False Positive Rate')
axes[2].set_title('Scheduling Aggressiveness\n(Lower is Better)')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('scheduler_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved as scheduler_comparison.png")

## Key Findings

**Entropy-Adaptive (proposed) vs Baseline:**

- **Completion Variance**: Lower CV indicates more predictable job completion times
- **Foreground Protection**: Lower P95 latency means user-facing tasks are less affected by background work
- **False Positives**: Lower rate means the scheduler correctly predicts when spikes are imminent

The entropy-adaptive approach combines two signals:
1. **Entropy** detects unpredictable load patterns and raises the scheduling threshold
2. **Momentum** detects load trends (rising/falling) and prioritizes scheduling during safe windows

**To scale this experiment:**
1. Increase `N_TRIALS` from 2 to 50 (in config cell)
2. Increase `TRIAL_DURATION_SEC` from 5 to 60
3. Run all cells again

The results should show clearer separation between schedulers as the sample size grows.